# SHIPP — Controlled DELETE Validation: Listing 002

**Test ID:** `delete-test-listing-002`

Use this notebook only for validation.

- Lakebase INSERT/DELETE SQL must run in the **Lakebase SQL editor**.
- Silver reconstruction/write must run in `10_silver_listings_dev.ipynb`.
- This notebook checks that the row exists in Silver before DELETE, then disappears after DELETE.


## Gate 1 — Configuration


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "bootcamp_students.shipp_bronze.lb_listings_history"
TARGET_TABLE = "bootcamp_students.shipp_silver.silver_listings"
BUSINESS_KEY = "listing_id"
DELETE_TEST_ID = "delete-test-listing-002"
TEST_ID = DELETE_TEST_ID

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("DELETE_TEST_ID =", DELETE_TEST_ID)


## Gate 2 — Capture baseline Silver


In [ ]:
baseline_silver_ids = {
    row[BUSINESS_KEY]
    for row in spark.table(TARGET_TABLE).select(BUSINESS_KEY).collect()
}

print("Baseline Silver count:", len(baseline_silver_ids))
print("Baseline Silver IDs:", sorted(baseline_silver_ids))

assert DELETE_TEST_ID not in baseline_silver_ids
print("PASS — Test ID is absent from baseline Silver.")


## Gate 3 — Create temporary listing in Lakebase

**WHERE:** Lakebase SQL editor

```sql
SELECT user_id
FROM shipp.users
WHERE user_id = 'demo-donor-001';

INSERT INTO shipp.listings (
    listing_id,
    donor_id,
    title,
    description,
    category,
    condition,
    location,
    latitude,
    longitude,
    available_from,
    available_until,
    status
)
VALUES (
    'delete-test-listing-002',
    'demo-donor-001',
    'DELETE CDC Test Listing 002',
    'Temporary listing created only to validate CDC delete semantics.',
    'FURNITURE',
    'GOOD',
    'Abu Dhabi',
    24.4976,
    54.4075,
    CURRENT_TIMESTAMP,
    CURRENT_TIMESTAMP + INTERVAL '7 days',
    'AVAILABLE'
);

SELECT listing_id, donor_id, title, status
FROM shipp.listings
WHERE listing_id = 'delete-test-listing-002';
```


## Gate 4 — Confirm INSERT reached Bronze


In [ ]:
bronze_df = spark.table(SOURCE_TABLE)

insert_history_df = (
    bronze_df
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "updated_at",
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

display(insert_history_df)

insert_count = insert_history_df.filter(
    F.col("_pg_change_type") == "insert"
).count()

print("Observed INSERT events:", insert_count)

assert insert_count >= 1, (
    "STOP: Bronze INSERT not observed yet. Wait for CDF and rerun."
)

print("PASS — Bronze contains the temporary INSERT event.")


## Gate 5A — Refresh Silver in `10_silver_listings_dev.ipynb`

**WHERE:** `10_silver_listings_dev.ipynb`

Run from **Step 1 through Step 12**.

Immediately after **Step 10** add this temporary diagnostic cell there:

```python
print(
    "delete-test-listing-002 in silver_df:",
    silver_df
    .filter(F.col("listing_id") == "delete-test-listing-002")
    .count()
)
```

Expected: `1`

Immediately after **Step 12** add this temporary diagnostic cell there:

```python
print(
    "delete-test-listing-002 in persisted Silver:",
    spark.table(TARGET_TABLE)
    .filter(F.col("listing_id") == "delete-test-listing-002")
    .count()
)
```

Expected: `1`

Then return to this notebook.


## Gate 5B — Confirm row exists in persisted Silver before DELETE


In [ ]:
before_delete_count = (
    spark.table(TARGET_TABLE)
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .count()
)

print("Temporary listing in Silver before delete:", before_delete_count)

assert before_delete_count == 1, (
    "STOP: temporary listing did not reach Silver before delete."
)

print("PASS — Temporary listing exists in Silver before delete.")


## Gate 6 — Delete temporary listing in Lakebase

**Only after Gate 5B passes.**

**WHERE:** Lakebase SQL editor

```sql
DELETE FROM shipp.listings
WHERE listing_id = 'delete-test-listing-002'
RETURNING listing_id;

SELECT *
FROM shipp.listings
WHERE listing_id = 'delete-test-listing-002';
```

Expected:
- DELETE returns `delete-test-listing-002`
- final SELECT returns `0 rows`


## Gate 6A — Confirm real Bronze DELETE


In [ ]:
bronze_df = spark.table(SOURCE_TABLE)

delete_history_df = (
    bronze_df
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "updated_at",
    )
    .orderBy(
        F.col("_pg_lsn").asc(),
        F.col("_sort_by").asc(),
    )
)

display(delete_history_df)

latest_test_event = (
    delete_history_df
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
    .first()
)

print("Latest change type:", latest_test_event["_pg_change_type"])
print("Latest LSN:", latest_test_event["_pg_lsn"])
print("Latest sort:", latest_test_event["_sort_by"])

assert latest_test_event["_pg_change_type"] == "delete", (
    "STOP: newest Bronze event is not delete."
)

print("PASS — Newest Bronze event is a real DELETE.")


## Gate 7 — Build delete-aware current-state reconstruction


In [ ]:
CURRENT_STATE_CHANGE_TYPES = {
    "insert",
    "update_postimage",
    "delete",
}

current_state_events_df = (
    spark.table(SOURCE_TABLE)
    .filter(
        F.col("_pg_change_type").isin(
            list(CURRENT_STATE_CHANGE_TYPES)
        )
    )
)

latest_window = (
    Window
    .partitionBy(BUSINESS_KEY)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

ranked_df = (
    current_state_events_df
    .withColumn(
        "_current_rn",
        F.row_number().over(latest_window)
    )
)

latest_events_df = ranked_df.filter(F.col("_current_rn") == 1)

deleted_latest_df = latest_events_df.filter(
    F.col("_pg_change_type") == "delete"
)

current_rows_df = latest_events_df.filter(
    F.col("_pg_change_type") != "delete"
)

print("Latest logical entities:", latest_events_df.count())
print("Latest deleted entities:", deleted_latest_df.count())
print("Current rows surviving for Silver:", current_rows_df.count())


## Gate 7B — Validate DELETE reconstruction decision


In [ ]:
test_latest_df = latest_events_df.filter(
    F.col("listing_id") == DELETE_TEST_ID
)

assert test_latest_df.count() == 1

test_latest_row = (
    test_latest_df
    .select("_pg_change_type", "_pg_lsn", "_sort_by")
    .first()
)

print(
    "Latest test event:",
    test_latest_row["_pg_change_type"],
    test_latest_row["_pg_lsn"],
    test_latest_row["_sort_by"],
)

assert test_latest_row["_pg_change_type"] == "delete"

test_current_count = (
    current_rows_df
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .count()
)

print(
    "Test listing remaining in current_rows_df:",
    test_current_count
)

assert test_current_count == 0

print(
    "PASS — newest DELETE event correctly removes the listing "
    "from reconstructed current state."
)


## Gate 8A — Apply delete-aware logic in `10_silver_listings_dev.ipynb`

**WHERE:** `10_silver_listings_dev.ipynb`

Use this order:

```text
Bronze
→ keep insert + update_postimage + delete
→ rank newest event per listing_id
→ rn = 1
→ if newest event = delete, exclude it
→ build silver_df from surviving rows
→ write Silver
```

Then rerun Step 1 through Step 12.


## Gate 8B — Confirm deleted listing is absent from persisted Silver


In [ ]:
deleted_test_count = (
    spark.table(TARGET_TABLE)
    .filter(F.col(BUSINESS_KEY) == DELETE_TEST_ID)
    .count()
)

print(
    "Deleted test listing remaining in Silver:",
    deleted_test_count
)

assert deleted_test_count == 0

print("PASS — Deleted listing is absent from persisted Silver.")


## Gate 9 — Confirm unrelated listings are unchanged


In [ ]:
final_silver_ids = {
    row[BUSINESS_KEY]
    for row in spark.table(TARGET_TABLE).select(BUSINESS_KEY).collect()
}

print("Baseline IDs:", sorted(baseline_silver_ids))
print("Final IDs:   ", sorted(final_silver_ids))

assert final_silver_ids == baseline_silver_ids

print("PASS — Unrelated Silver listings remain unchanged.")


## Gate 10 — Confirm every latest DELETE entity is excluded


In [ ]:
deleted_latest_ids_df = (
    latest_events_df
    .filter(F.col("_pg_change_type") == "delete")
    .select(BUSINESS_KEY)
    .distinct()
)

deleted_rows_still_in_silver = (
    spark.table(TARGET_TABLE)
    .join(
        deleted_latest_ids_df,
        on=BUSINESS_KEY,
        how="inner",
    )
    .count()
)

print(
    "Latest-delete entities still present in Silver:",
    deleted_rows_still_in_silver
)

assert deleted_rows_still_in_silver == 0

print("PASS — Latest DELETE entities are excluded from Silver.")


## Gate 11A — Capture Run 1 logical state


In [ ]:
logical_columns = [
    col
    for col in spark.table(TARGET_TABLE).columns
    if col != "silver_processed_at"
]

delete_run1_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

print("DELETE validation Run 1 rows:", len(delete_run1_rows))
print("PASS — Run 1 logical state captured.")


## Gate 11B — Rerun Silver once

**WHERE:** `10_silver_listings_dev.ipynb`

Rerun the delete-aware Silver pipeline once more without changing Lakebase data.

Then return here.


## Gate 11C — Compare Run 1 vs Run 2


In [ ]:
delete_run2_rows = [
    row.asDict(recursive=True)
    for row in (
        spark.table(TARGET_TABLE)
        .select(*logical_columns)
        .orderBy(BUSINESS_KEY)
        .collect()
    )
]

print("DELETE validation Run 1 rows:", len(delete_run1_rows))
print("DELETE validation Run 2 rows:", len(delete_run2_rows))

assert delete_run1_rows == delete_run2_rows

assert DELETE_TEST_ID not in {
    row[BUSINESS_KEY]
    for row in delete_run2_rows
}

print("PASS — DELETE-aware Silver rerun is logically idempotent.")


## Final acceptance

Mark `LISTING DELETE → VALIDATED` only when:

- Bronze INSERT is observed
- test listing exists in Silver before DELETE
- Bronze DELETE is observed
- DELETE is newest by `_pg_lsn`, `_sort_by`
- `current_rows_df` excludes the deleted listing
- persisted Silver excludes the deleted listing
- unrelated listings are unchanged
- rerun is logically idempotent
